# Wireless Networking and Mobile Computing

### 1. LED-Based Visible Light Protocol

This VLC system uses a single LED for both transmission and reception. The LED always appears illuminated to humans, while data is embedded in rapid brightness changes. Communication relies on precise slot timing, idle synchronization, and a lightweight CSMA/CA MAC similar in spirit to RF systems but adapted for light.


#### 1.1 Idle Mode and Constant Illumination

The LED must appear continuously lit for human perception (e.g., toys, lamps).
Even when idle, the system transmits a periodic pattern. This pattern serves two purposes:

* maintains constant brightness for illumination
* provides timing reference for receivers

Devices continuously watch this idle pattern to stay synchronized.


<center>
  <img src="../../Admin/light/vlc_idleMode.png" width="50%">
</center>


#### 1.2 Slot Synchronization

Communication uses time-slotted signaling.
Every device aligns to shared OFF-slots , if timing drifts, communication fails.

* receivers detect the idle pattern
* align their internal clock to the pattern
* confirm OFF slots match expected timing
* enables fair access and collision avoidance

<center>
  <img src="../../Admin/light/vlc_sync.png" width="34%">
  <img src="../../Admin/light/vlc_nosync.png" width="30%">
</center>

Correct alignment → packets decode.
Misalignment → pulses land in the wrong slots → no reception.


#### 1.3 Transmit and Receive Modes

The system supports two operating modes:

**TX mode** (active transmission)

* uses **2-pulse position modulation (2-PPM)**
* each bit occupies two time slots
* **exactly one ON and one OFF slot per bit**
  → constant average brightness
  → works even while “shining”

**RX mode** (listening)

* LED reverse-biased
* LED acts as light sensor
* microcontroller samples voltage at end of each slot
* detects ON vs OFF pulses

<center>
  <img src="../../Admin/light/tx_rx_mode.png" width="60%">
</center>

Bit mapping:

| Bit | Slot 1 | Slot 2 |
| --- | ------ | ------ |
| 0   | ON     | OFF    |
| 1   | OFF    | ON     |

Signal stays perceptually constant but carries binary data.

#### 1.4 Frame Exchange

Communication follows `DATA → ACK → return to idle sync`.

Basic sequence:

1. Device 1 sends DATA frame
2. Device 2 checks frame (CRC)
3. If valid, device 1 transmits ACK
4. Devices resynchronize to idle pattern
5. Device 1 sends next DATA if pending

<center>
  <img src="../../Admin/light/light_frameExchange.png" width="70%">
</center>


#### 1.5 MAC: CSMA/CA for Light

The system adapts a CSMA/CA strategy:

* devices listen to idle pattern
* if channel idle → random backoff slots
* transmit if slots remain idle
* ACK frames get priority (minimum CW = 1 slot)

Receivers perform **preamble (SFD)** detection.
If SFD recognized → switch to RX and block TX until frame ends.
If no consistent modulation → channel considered idle.

<center>
  <img src="../../Admin/light/vlc_csma.png" width="60%">
</center>

Channel sensing rule:
Small variation = idle
Pattern detected = busy


#### 1.6 Retransmission Behavior

Collisions or errors trigger retransmission:

* corrupted frame → CRC fail → **no ACK**
* sender waits for ACK timeout
* retries transmission

This keeps the system robust in multi-device environments and under weak light or reflection.

<center>
  <img src="../../Admin/light/vlc_retransmit.png" width="55%">
</center>
